# Transformer propio (encoder temporal): proceso

Cuaderno de proceso del Transformer escrito para este proyecto. Las celdas de código son lecturas
ligeras (no entrenan nada) y avisan si falta algún archivo. Modelos hermanos:
[`01_xgboost_booster.ipynb`](01_xgboost_booster.ipynb), [`02_tcn.ipynb`](02_tcn.ipynb),
[`03_cnn_lstm.ipynb`](03_cnn_lstm.ipynb) y [`05_timesfm3.ipynb`](05_timesfm3.ipynb).

> **Estado de la validación.** Optuna (20 pruebas, 12 cortes de ajuste) y la validación de 24
> cortes con `--folds` están terminadas, en CPU y con 3 semillas. La sección 6 compara con los
> demás modelos y la 7 recoge las limitaciones.

In [ ]:
import os
import warnings
from pathlib import Path

import polars as pl
import yaml

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Raiz del repositorio:", ROOT)


def leer_csv(ruta):
    try:
        return pl.read_csv(ruta)
    except FileNotFoundError:
        print(f"No existe {ruta}. Ver la seccion 'Como reproducir'.")
    except Exception as error:
        print(f"No se pudo leer {ruta}: {error}")


def leer_yaml(ruta):
    with open(ruta, encoding="utf-8") as archivo:
        return yaml.safe_load(archivo)

## 1. Objetivo y criterio de selección

El mismo de los otros cuadernos: pronosticar el nivel en **Ciudad Bolívar** y **Palúa** a 1–14
días, reportando los **días 7 y 14**, con el criterio fijado antes de ver resultados (`rel_mae < 1`
en cada estación y, entre los que cumplen, el menor MAE medio).

**Qué pregunta responde este modelo.** Si la **atención** (cada día de la ventana mira a todos los
demás, sin recorrerla en orden como la LSTM ni con un campo receptivo fijo como la TCN) aporta
algo con ~5000 días de entrenamiento. No es el TFT ni un modelo de librería: solo se usa
`nn.TransformerEncoderLayer` de PyTorch; la ventana, el escalado, el entrenamiento y la
reconstrucción de niveles son los del CNN-LSTM.

## 2. Datos y partición

Los mismos que el resto: hoja `Entrenar` de `DATOS RIO ORINOCO.xlsx`, **2009-01-01 a 2024-12-31**,
el frame de `load_history` con las cuatro estaciones, **24 cortes** de 2021-01-26 a 2022-12-17
cada 30 días y el **test (2023 en adelante) sin tocar**. Ver las secciones 2 y 3 de
[`03_cnn_lstm.ipynb`](03_cnn_lstm.ipynb); no se repiten aquí.

## 3. Arquitectura

Código: [`models/neural/transformer.py`](../src/orinoco_forecast/models/neural/transformer.py)
(`TemporalTransformer` y `TransformerRunner`).

```text
ventana (lote, input_size, variables)
  -> Linear(variables, d_model)              # cada día a un vector
  -> + posición aprendida (1, input_size, d_model)
  -> num_layers x TransformerEncoderLayer    # atención + feed-forward, pre-norm
  -> último día de la ventana
  -> Linear(d_model, 14)                     # los 14 pasos a la vez
```

La celda siguiente construye la red de la configuración ajustada con datos aleatorios y cuenta
sus parámetros.

In [ ]:
try:
    import torch

    from orinoco_forecast.models.neural.transformer import TemporalTransformer

    cfg = leer_yaml("configs/models/transformer.yaml")
    red = TemporalTransformer(n_features=8, horizon=14, input_size=cfg["input_size"],
                              d_model=cfg["d_model"], nhead=cfg["nhead"],
                              num_layers=cfg["num_layers"], dim_feedforward=cfg["dim_feedforward"])
    salida = red(torch.randn(4, cfg["input_size"], 8))
    print("Salida:", tuple(salida.shape))
    print("Parametros:", sum(p.numel() for p in red.parameters()))
except Exception as error:
    print("No se pudo construir la red:", error)

## 4. Diseño del runner

`TransformerRunner` **hereda de `CnnLstmRunner`** y solo cambia la red (`_build_net`) y el prefijo
de las columnas (`TRANSFORMER_s<semilla>`). Así el Transformer y el CNN-LSTM se diferencian
únicamente en la arquitectura.

| Decisión | Qué problema resuelve | Cómo funciona por dentro | Alternativa descartada y por qué | Cuándo sería un error |
|---|---|---|---|---|
| **Heredar el runner del CNN-LSTM** | Comparar arquitecturas con el mismo pipeline | Ventanas, escalado robusto por ventana, `target_transform: difference`, parada temprana y semillas idénticos | Un runner nuevo: duplicaría ~200 líneas y cualquier diferencia se mezclaría con la arquitectura | Si el Transformer necesitara otro preprocesado |
| **Posición aprendida** | La atención no sabe en qué orden van los días | Un vector por posición de la ventana, sumado a la entrada | Senos y cosenos fijos: no ganan nada con ventanas de tamaño fijo | Si la ventana cambiara de tamaño en inferencia |
| **Pre-norm** (`norm_first=True`) | Entrenamiento más estable con pocos datos | Normaliza antes de cada subcapa | Post-norm (el original): más sensible a la tasa de aprendizaje | — |
| **Salida desde el último día** | Resumir la ventana en un vector | El encoder ya mezcló toda la ventana en cada posición; se toma la última | Promedio de todas las posiciones o un token `[CLS]`: más código sin evidencia de mejora | — |
| **14 pasos a la vez** (MIMO) | Evitar acumular errores | Una capa lineal a 14 salidas | Autorregresivo: propaga el error día a día | — |
| **Encoder sin máscara causal** | — | La ventana es toda pasada, así que mirar "adelante" dentro de ella no filtra el futuro | Máscara causal: limitaría sin motivo | Si la ventana incluyera días posteriores al corte (un test lo descarta) |
| **Mismo `drop_covariates: [rain_, oni]`** | La lluvia de los pueblos empeoró las redes | Oculta lluvia y ONI | Pasarle todo | — |

In [ ]:
try:
    cfg = leer_yaml("configs/models/transformer.yaml")
    for clave in ["target_transform", "input_size", "d_model", "nhead", "num_layers",
                  "dim_feedforward", "dropout", "learning_rate", "batch_size", "max_epochs",
                  "val_size", "early_stop_patience", "drop_covariates", "seeds"]:
        print(f"{clave:22s} {cfg.get(clave)}")
except FileNotFoundError:
    print("No existe configs/models/transformer.yaml")

## 5. Optuna: 20 pruebas

Igual que el CNN-LSTM ([`scripts/tune_transformer.py`](../scripts/tune_transformer.py), que
reutiliza `run_study` de `tune_cnn_lstm.py`): 12 cortes de ajuste de 2020, un ajuste por prueba,
objetivo `rel_mae` medio entre estaciones. Espacio: `input_size` {28, 56, 112}, `d_model`
{16, 32, 64}, `nhead` {2, 4}, `num_layers` 1–3, `dim_feedforward` {32, 64, 128}, `dropout` 0–0,3,
tasa de aprendizaje 3e-4–1e-2 (log) y lote {128, 256}.

**Resultado:** gana la prueba 18 (`rel_mae` 0,675: Ciudad Bolívar 0,44, Palúa 0,91) con
`input_size` 112, `d_model` 32, 2 cabezas, 2 capas, `dim_feedforward` 64 y `dropout` 0,12.

**Dos cautelas al leer la tabla:**

- **La búsqueda no convergió.** Las cuatro mejores pruebas son las últimas (16 a 19) y todas usan
  la ventana más larga (112). Con más pruebas, o con `input_size` mayor que 112, probablemente
  mejore. No se amplió para mantener el mismo presupuesto (20) que los otros modelos.
- **El denominador de `rel_mae` es la persistencia del primer corte de ajuste**, no la de cada
  corte: `NaiveRunner.predict` ignora el `cutoff` y `tune_cnn_lstm.py` lo ajusta una sola vez. El
  orden de las pruebas no cambia (el denominador es el mismo para todas), pero el valor no es un
  `rel_mae` real y no se compara con la sección 6.

In [ ]:
pruebas = leer_csv("reports/tuning/transformer_trials.csv")
if pruebas is not None:
    with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=220, float_precision=3):
        print(pruebas.sort("value").select("number", "value", "rel_mae_ciudad_bolivar", "rel_mae_palua",
                                           "params_input_size", "params_d_model", "params_nhead",
                                           "params_num_layers", "params_dropout",
                                           "params_learning_rate").head(8))

## 6. Validación de 24 cortes (la que cuenta para elegir)

Configuración ajustada, CPU, media de las 3 semillas contra Naive (`MAE / rel_mae`; `p` = rango
del test de signos entre semillas):

| Estación | Horizonte | Naive | Transformer | p |
|---|---|---|---|---|
| Ciudad Bolívar | día 7 | 0,526 | 0,272 / 0,52 | 0,007–0,064 |
| Ciudad Bolívar | día 14 | 1,064 | 0,522 / 0,49 | 0,002–0,064 |
| Ciudad Bolívar | días 1 a 14 | 0,570 | 0,294 / 0,52 | 0,007–0,152 |
| Palúa | día 7 | 0,435 | 0,375 / 0,86 | 0,541–1,000 |
| Palúa | día 14 | 0,891 | 0,608 / 0,68 | 0,007–0,152 |
| Palúa | días 1 a 14 | 0,488 | 0,357 / 0,73 | 0,023–0,307 |

MAE medio entre estaciones (días 1 a 14): **Transformer 0,326 m**, frente a TimesFM 3 0,300,
XGBoost 0,309, TCN 0,317, CNN-LSTM 0,350 y ARIMAX 0,351. Sin ajustar (configuración inicial,
Colab) daba 0,330 (Ciudad Bolívar 0,310, Palúa 0,350).

**Lectura.**

- **Cumple el criterio** (`rel_mae < 1` en las dos estaciones y los tres horizontes) y mejora al
  CNN-LSTM con el mismo pipeline: la atención aporta frente a convolución + LSTM.
- **No es el mejor en promedio**: queda cuarto, detrás de TimesFM 3, XGBoost y TCN. Pero en
  **Ciudad Bolívar a 7 días** tiene el menor MAE de todos (0,272, frente a 0,274 del TCN y 0,278
  de TimesFM 3); la diferencia es mínima frente a la dispersión entre cortes.
- **El ajuste ayudó solo en Ciudad Bolívar** (0,310 → 0,294); en Palúa empeoró algo
  (0,350 → 0,357). El objetivo de Optuna mezcla las dos estaciones y la prueba ganadora es mucho
  mejor en Ciudad Bolívar (0,44) que en Palúa (0,91).
- **Palúa a 7 días no supera a Naive con evidencia** (p ≥ 0,54, gana en la mitad de los cortes),
  el mismo problema que el booster y el TCN.
- **Varianza entre semillas**: la semilla 3 da 0,269 en Ciudad Bolívar y las otras dos 0,307.

In [ ]:
def mae_por_modelo(ruta, nombre):
    try:
        tabla = pl.read_parquet(ruta)
    except FileNotFoundError:
        print(f"No existe {ruta}")
        return None
    return (
        tabla.filter((pl.col("metric") == "mae") & pl.col("horizon").cast(pl.String).is_in(["7", "14", "all"]))
        .group_by("unique_id", "horizon")
        .agg(pl.lit(nombre).alias("modelo"), pl.col("mean_model").mean().alias("mae"),
             pl.col("std_model").mean().alias("desv_cortes"), pl.col("sign_test_p").min().alias("p"))
    )


fuentes = {
    "transformer": "reports/predictions/transformer/backtest/naive/comparison.parquet",
    "booster": "reports/predictions/booster/backtest/naive/comparison.parquet",
    "tcn": "reports/predictions/tcn/backtest/naive/comparison.parquet",
    "cnn_lstm": "reports/predictions/cnn_lstm/backtest/naive/comparison.parquet",
    "arimax": "reports/predictions/arimax/backtest/naive/comparison.parquet",
    "timesfm3": "reports/predictions/timesfm3/backtest/naive/comparison.parquet",
}
partes = [t for n, r in fuentes.items() if (t := mae_por_modelo(r, n)) is not None]
if partes:
    with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=200, float_precision=3):
        print(pl.concat(partes).sort("unique_id", "horizon", "mae"))

## 7. Limitaciones

- **Búsqueda sin converger** (sección 5): 20 pruebas y el óptimo en el borde del espacio
  (`input_size` 112).
- **Objetivo de ajuste conjunto.** Una sola configuración para las dos estaciones; Palúa
  probablemente pide otra.
- **CPU, unas 7 h** para los 24 cortes con 3 semillas y la CPU compartida. En GPU los números
  pueden diferir algo (la GPU no es determinista).
- **Sin lluvia ni ONI** (`drop_covariates`); no se probó la lluvia de las subcuencas
  (`feat/basin-rainfall`).
- **Pocos datos para atención**: ~5000 días; los Transformers suelen pedir más.
- **Test final sin tocar.**

## 8. Cómo reproducir

```bash
# Genera el parquet (una vez)
uv run python scripts/run_model.py booster --write-dataset
# Optuna (20 pruebas; estudio en artifacts/optuna/transformer_2009.db)
uv run python scripts/tune_transformer.py --trials 20
# Un corte contra Naive
uv run python scripts/run_model.py transformer --against naive
# 24 cortes (salidas en data/predictions/transformer/backtest/naive/)
uv run python scripts/run_model.py transformer --against naive --folds
```

En Colab: [`colab_folds.ipynb`](colab_folds.ipynb) con `MODEL = "transformer"` (y `TUNE = True`
para la búsqueda).